# 07b - The combination: TMD-style projection + adversarial training

**What this is.** A hybrid of the two kinds of defense in the paper's Table 1: TMD's manifold projection (learned, as in notebooks 05-06) and adversarial training (the family of the paper's ASCC and DNE baselines). Unlike TMD, this defense learns from attacks. So it is compared with TMD *and* with the paper's adversarial training baselines, and has to be described as an adversarially trained defense.

**Rules (as in notebooks 05-06).** Attacks are called as black boxes on *training* sentences, and the defense learns from what they produce. Nothing is copied from the attack code: no word lists, budgets or search details. Validation sentences are used only to choose between variants, and test sentences only for the final run. One thing from 05-06 is dropped: the synonym list used by the encoder's denoising term is replaced by notebook 07's generic noise.

**What is new compared with notebook 06 (`enc_adv4`):**
1. **BERT itself is adversarially trained**, not only the projection. In 05-06, BERT stayed frozen, so the encoder had to undo whatever a few swapped words did to the sentence embedding, and the rounds stopped helping (`enc_adv5` added nothing). Here BERT trains on the ~5,200 attacked training sentences that notebooks 05-06 already saved, plus notebook 07's noise views. It is then attacked again on fresh training sentences and trained once more.
2. **The projection encoder is retrained on the new BERT**, with one adaptive round against the full new defense. It is also offered with **weight averaging** across epochs, instead of one kept checkpoint (`enc_adv4` and `enc_adv5` were 11.5 points apart on validation TextFooler).
3. It **starts from notebook 07's noise-robust BERT** if that BERT beat vanilla BERT by at least 3 points on validation. Otherwise it starts from notebook 00's BERT.

**Variants**, all on the new BERT `advbert2`:

| tag | projection |
|---|---|
| `advbert2` | none: the adversarially trained BERT alone |
| `advbert2_tmd` | the paper's TMD, Table 6 GAN |
| `advbert2_enc1` | encoder trained on the saved attacked sentences |
| `advbert2_enc2` | after one adaptive round against `advbert2_enc1` |
| `advbert2_enc2_swa` | the same, weights averaged over the second half of training |

Notebook 06's `enc_adv4_m9` stays in the running.

**Selection protocol (fixed before any result):** the same 200 validation sentences as notebook 06. The winner has the highest mean of TextFooler and BERT-Attack among variants within 1 point of our TMD's clean validation accuracy. Only that variant is attacked on the 1000 test sentences, once. If `enc_adv4_m9` wins, notebook 06's test result stands and nothing new is run on test.

**Cost (L4, rough):** about 6 hours. That is roughly 1 h of BERT training, 2 h of attacks on training sentences for the two rounds, 45 min of validation attacks and 1 h for the test run. Everything is cached on Drive and resumes after a disconnect. Run notebooks 00, 05, 06 and 07 first.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import os, json, torch, pandas as pd, tmd_lab as T
T.setup_env()                      # the authors' TextAttack fork for every attack
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # same folder as notebooks 00-07
N_VAL = 200                        # validation sentences per attack (the same 200 as notebooks 06 and 07)
N_TEST = 1000                      # the single final test run
N_ROUND = 1000                     # training sentences per attack in each new adversarial round
SCREEN_ATTACKS = ('textfooler', 'bertattack')
ALL = ('textfooler', 'bertattack', 'pwws')
ENC_EPOCHS = 30
# Encoder losses as in notebooks 05-06; the denoising term now uses notebook 07's generic noise, not synonyms
W_DEN = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=0.0, w_cls=0.0)
W_ADV = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=1.0, w_cls=1.0)

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
gan = lab.get_gan('base', E_tr, E_va)

def load_enc(name, g):
    enc = T.ProjEncoder(g.latent_dim, g.code_dim).to(T.DEV)
    enc.load_state_dict(torch.load(lab._path(f'projenc_{name}_L{g.latent_dim}C{g.code_dim}.pt')))
    return enc.eval()

refs = {'baseline': T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K),          # our TMD
        'enc_adv4_m9': T.Victim(lab, gan=gan, enc=load_enc('enc_adv4', gan), mode='tmd', proj_mode='encoder')}

START, RATE = 'base', (0.1, 0.3)
choice = lab._path('nb07_choice.json')
if os.path.exists(choice):
    c = json.load(open(choice))
    print(f"notebook 07: {c['bert']} was {c['gain_over_vanilla']:+.1f} points vs vanilla BERT on validation")
    if c['gain_over_vanilla'] >= 3.0:
        START, RATE = c['bert'], tuple(c['rate'])
else:
    print('notebook 07 has not been run: starting from notebook 00\'s BERT')
print('adversarial training starts from:', START)
noise = T.NoiseAugmenter(lab.train_ds['text'], rate=RATE, seed=1)

## Step 1: adversarially train BERT on the attacked training sentences saved by notebooks 05-06
Each step: 32 clean training sentences with their noise views, plus 16 (clean, attacked) training pairs. Cross-entropy on all of them, plus the two consistency terms of notebook 07 for every (clean, changed) pair.

In [ ]:
adv_tr = T.adversarial_texts(lab, 'train', exclude=('advbert',))   # attacks on TRAINING sentences from 05-06
assert adv_tr, 'no attack logs (advpairs_*.jsonl) found in the Drive folder: notebooks 05 and 06 must have run there'
print(len(adv_tr), 'attacked training sentences from notebooks 05-06')
lab.finetune_robust('advbert1', init=START, epochs=2, lr=1e-5, noise_fn=noise, adv_texts=adv_tr)

## Step 2: attack the new BERT on fresh training sentences, then train once more
Three black-box attacks on 1000 new training sentences each, against `advbert1` itself, so the next epoch learns from what still fools it.

In [ ]:
lab.use_model('advbert1')
v1 = T.Victim(lab, mode='none')
for attack, seed in (('textfooler', 31), ('pwws', 32), ('bertattack', 33)):
    T.adversarial_pairs(lab, v1, 'advbert1', attack, N_ROUND, seed)
adv_tr = T.adversarial_texts(lab, 'train', exclude=('advbert2',))  # 05-06 + the new round
print(len(adv_tr), 'attacked training sentences in total')
lab.finetune_robust('advbert2', init='advbert1', epochs=1, lr=1e-5, noise_fn=noise, adv_texts=adv_tr)

## Step 3: projections on the new BERT
New embeddings and a GAN with the paper's Table 6 hyperparameters. Every saved attacked training sentence is re-embedded with the new BERT to train the encoder. The best epoch is chosen on the attacked *validation* sentences from notebook 06.

In [ ]:
lab.use_model('advbert2')
Ea_tr, _ = lab.embeddings('train', 'advbert2')
Ea_va, _ = lab.embeddings('val', 'advbert2')
Ea_te, _ = lab.embeddings('test', 'advbert2')
Ea_noise, _ = lab.embeddings('train', 'advbert2', augment_fn=noise, name='train_noise')   # generic noise, no synonyms
gan_a = lab.get_gan('advbert2', Ea_tr, Ea_va)
head_a = lab.model.classifier

adv_a = (T.embed_texts(lab, [r[1] for r in adv_tr]), T.embed_texts(lab, [r[0] for r in adv_tr]),
         torch.tensor([r[2] for r in adv_tr]))
val_rows = T.adversarial_texts(lab, 'val')            # attacks on VALIDATION sentences (notebook 06), never test
val_adv_a = (T.embed_texts(lab, [r[1] for r in val_rows]), torch.tensor([r[2] for r in val_rows]))
print(len(adv_a[0]), 'training pairs,', len(val_adv_a[0]), 'attacked validation sentences for choosing epochs')

def get_enc(name, **kw):
    # Train an encoder on the new BERT, or load it from Drive. With swa_from it also returns the averaged encoder.
    paths = [lab._path(f'projenc_{name}_L{gan_a.latent_dim}C{gan_a.code_dim}.pt')]
    if kw.get('swa_from'):
        paths.append(paths[0].replace('.pt', '_swa.pt'))
    if all(os.path.exists(p) for p in paths):
        encs = []
        for p in paths:
            e = T.ProjEncoder(gan_a.latent_dim, gan_a.code_dim).to(T.DEV)
            e.load_state_dict(torch.load(p))
            encs.append(e.eval())
        print('loaded', name)
    else:
        print('training', name)
        out = T.train_proj_encoder(gan_a, Ea_tr, Y_tr, Ea_va, Y_va, E_aug=Ea_noise, head=head_a, epochs=ENC_EPOCHS, **kw)
        encs = list(out) if isinstance(out, tuple) else [out]
        for e, p in zip(encs, paths):
            torch.save(e.state_dict(), p)
    return encs if len(encs) > 1 else encs[0]

def enc_victim(e):
    return T.Victim(lab, gan=gan_a, enc=e, mode='tmd', proj_mode='encoder')

enc_den = get_enc('b_enc_den', **W_DEN)
enc1 = get_enc('b_enc1', adv=adv_a, init=enc_den, select='best', val_adv=val_adv_a, **W_ADV)

## Step 4: one adaptive round against the full new defense
Black-box attacks on 1000 new training sentences each, against `advbert2` + `enc1`. The next encoder starts from `enc1` and learns from all pairs. It keeps the best epoch on validation and also returns the weight-averaged encoder.

In [ ]:
new = [T.adversarial_pairs(lab, enc_victim(enc1), 'advbert2_enc1', a, N_ROUND, s)
       for a, s in (('textfooler', 34), ('bertattack', 35), ('pwws', 36))]
adv_a2 = tuple(torch.cat([adv_a[j]] + [p[j] for p in new]) for j in range(3))
print(len(adv_a2[0]), 'training pairs after the adaptive round')
enc2, enc2_swa = get_enc('b_enc2', adv=adv_a2, init=enc1, select='best', val_adv=val_adv_a,
                         swa_from=ENC_EPOCHS // 2, **W_ADV)

## Screening on validation sentences
The same 200 validation sentences as notebooks 06 and 07. Nothing here touches the test set. Rows from earlier notebooks are read from Drive.

In [ ]:
lab.use_model('advbert2')
variants = {'advbert2': T.Victim(lab, mode='none'),
            'advbert2_tmd': T.Victim(lab, gan=gan_a, mode='tmd', proj_mode='sample', k=BASE_K),
            'advbert2_enc1': enc_victim(enc1), 'advbert2_enc2': enc_victim(enc2), 'advbert2_enc2_swa': enc_victim(enc2_swa)}
for tag, v in variants.items():
    T.evaluate(lab, v, tag, Ea_va, Y_va, attacks=SCREEN_ATTACKS, n=N_VAL, split='val', need_log=True)

earlier = ['vanilla', 'baseline', 'tmd_t6', 'enc_adv3_m9', 'enc_adv4_m9']
if os.path.exists(choice):
    earlier += [c['bert'], c['bert'] + '_tmd']
table = T.screen(lab, earlier + list(variants), N_VAL)
display(table)

In [ ]:
# Selection rule (fixed in advance): best validation mean of TextFooler and BERT-Attack among the attack-trained
# variants (the new ones and notebook 06's enc_adv4_m9), within 1 point of our TMD's clean validation accuracy.
base = table[table.variant == 'baseline'].iloc[0]
cand = table[table.variant.isin(list(variants) + ['enc_adv4_m9']) & (table.clean >= base['clean'] - 1.0)]
assert len(cand), 'no candidate keeps clean accuracy within 1 point of TMD: stop here and send the table'
chosen = cand.loc[cand['mean'].idxmax(), 'variant']
row = cand[cand.variant == chosen].iloc[0]
print('chosen variant:', chosen)
print(f"vs our TMD on validation: TextFooler {row['TextFooler'] - base['TextFooler']:+.1f}, "
      f"BERT-Attack {row['BERT-Attack'] - base['BERT-Attack']:+.1f}")
display(cand.sort_values('mean', ascending=False))

## The one test run
The chosen variant on the 1000 test sentences (all three attacks), against the paper's Table 1 (TMD and the best baseline, which for an adversarially trained defense is the like-for-like comparison). It is also compared with our TMD and with notebook 06's `enc_adv4` (McNemar tests, same sentences), and in the transfer test. Run this once.

In [ ]:
if chosen == 'enc_adv4_m9':
    print("notebook 06's enc_adv4 is still the best on validation: its test result from notebook 06 stands")
    display(T.compare(lab, 'enc_adv4_m9', baseline_tag='baseline', n=N_TEST))
else:
    v = variants[chosen]
    T.evaluate(lab, v, chosen, Ea_te, Y_te, attacks=ALL, n=N_TEST, need_log=True)
    display(T.compare(lab, chosen, baseline_tag='baseline', n=N_TEST))
    display(pd.DataFrame([T.mcnemar(lab, ref, chosen, a, N_TEST) for ref in ('baseline', 'enc_adv4_m9') for a in ALL]))
    for attack in ALL:
        print('transfer from vanilla BERT,', attack)
        display(T.transfer_eval(lab, {**refs, chosen: v}, 'vanilla', attack, N_TEST))

## After the test run: a second training seed, and the transfer and worst-case numbers for notebook 09
Notebook 08 showed that one training seed can mislead (enc_adv4's second seed was 15 points lower on TextFooler). So the chosen encoder is retrained with another seed on the same data and rules, and attacked once on test with TextFooler and BERT-Attack. The second cell computes transfer and per-sentence worst case for the chosen defense and saves them next to notebook 08's results for notebook 09. Both are cached and resume after a disconnect.

In [ ]:
if chosen == 'advbert2_enc1':
    P = lab._path(f'projenc_b_enc1_s7_L{gan_a.latent_dim}C{gan_a.code_dim}.pt')
    enc1_s7 = T.ProjEncoder(gan_a.latent_dim, gan_a.code_dim).to(T.DEV)
    if os.path.exists(P):
        enc1_s7.load_state_dict(torch.load(P)); enc1_s7.eval()
        print('loaded the second-seed encoder')
    else:
        den_s7 = T.train_proj_encoder(gan_a, Ea_tr, Y_tr, Ea_va, Y_va, E_aug=Ea_noise, head=head_a,
                                      epochs=ENC_EPOCHS, seed=7, **W_DEN)
        enc1_s7 = T.train_proj_encoder(gan_a, Ea_tr, Y_tr, Ea_va, Y_va, E_aug=Ea_noise, head=head_a,
                                       epochs=ENC_EPOCHS, adv=adv_a, init=den_s7, select='best', val_adv=val_adv_a,
                                       seed=7, **W_ADV)
        torch.save(enc1_s7.state_dict(), P)
    lab.use_model('advbert2')
    T.evaluate(lab, enc_victim(enc1_s7), 'advbert2_enc1_s7', Ea_te, Y_te, attacks=('textfooler', 'bertattack'),
               n=N_TEST, need_log=True)
    display(T.screen(lab, ['advbert2_enc1', 'advbert2_enc1_s7'], N_TEST, split='test',
                     attacks=('textfooler', 'bertattack')))
else:
    print(f'this cell is written for advbert2_enc1; the chosen variant is {chosen}, so it is skipped')

In [ ]:
if chosen != 'enc_adv4_m9':
    RES = lab._path('nb08_results.json')
    res = json.load(open(RES)) if os.path.exists(RES) else {}
    v = variants[chosen]
    rows = [r for r in res.get('worst_case', []) if r['defense'] != chosen]
    for attack in ALL:
        src = [r for r in map(json.loads, open(T.examples_path(lab, 'vanilla', N_TEST, attack))) if r['status'] == 'S']
        y, texts = torch.tensor([r['label'] for r in src]), [r['adv_text'] for r in src]
        pred = torch.cat([v.scores(texts[i:i + 64]).argmax(-1).cpu() for i in range(0, len(texts), 64)])
        ok = {r['i']: bool(p) for r, p in zip(src, pred == y)}
        adapt = {r['i']: r['status'] for r in map(json.loads, open(T.examples_path(lab, chosen, N_TEST, attack)))
                 if r['status'] != 'D'}
        n = len(adapt)
        rows.append(dict(attack=attack, defense=chosen,
                         adaptive=round(100 * sum(s == 'F' for s in adapt.values()) / n, 1),
                         transfer=round(100 * (pred == y).float().mean().item(), 1), transfer_n=len(src),
                         worst_case=round(100 * sum(s == 'F' and ok.get(i, True) for i, s in adapt.items()) / n, 1)))
    res['worst_case'] = rows
    json.dump(res, open(RES, 'w'), indent=1)
    display(pd.DataFrame([r for r in rows if r['defense'] == chosen]))